# Using SUPREME with your own components

This notebook takes the perspective of an **external user**: you `pip install`
SUPREME into your own project and plug in your **own datasets, models, unlearning
methods, baselines and metrics** - without editing any framework code.

SUPREME is registry-based. A component is resolved by *name* through a convention
(`Foo` -> `supreme.<subpackage>.Foo`), and you extend it by **registering a module
path** that points at *your* code. Resolution order is **runtime overrides ->
packaging entry points -> built-in convention**, so your components never collide
with or alter the built-ins. The full component interfaces and the Lightning Fabric
integration rules live in [`docs/extending.md`](../docs/extending.md).

## 1. Install SUPREME into your project

In your own project's environment (PyPI distribution name is `supreme-unlearning`;
you still `import supreme`):

```bash
pip install supreme-unlearning            # core (CPU / MPS)
pip install "supreme-unlearning[cuda]"    # + DeepSpeed / bitsandbytes / NVIDIA telemetry
```

`import supreme` is lightweight - it does **not** import the PyTorch/Lightning stack,
so registration works even before the heavy dependencies are touched.

In [ ]:
# Optional: install SUPREME if it isn't already in this kernel's environment.
# Use the %pip magic (not !pip) so it installs into THIS notebook's kernel, not
# a different Python on PATH. On Linux + NVIDIA use "supreme-unlearning[cuda]".
# Skips quickly if the requirement is already satisfied.
%pip install supreme-unlearning

In [ ]:
import supreme

print("SUPREME", supreme.__version__)
print("register API:", [n for n in supreme.__all__ if n.startswith("register")])

## 2. A guaranteed-runnable proof (no GPU needed)

Registration and resolution are torch-free, so this cell runs anywhere. We write a
tiny external module, register a method and a model from it, and confirm SUPREME
resolves the names to *our* module path and can load them. (Real components are
torch-based - that's section 3.)

In [ ]:
import importlib


# A guaranteed-runnable proof. Define two trivial components right here (no torch),
# register the OBJECTS directly, and watch the framework resolve + call them.
def acme_method(fabric=None, model=None, **kwargs):
    # A real method returns the unlearned model; this stub returns a marker so we can
    # prove resolution + invocation without the heavy stack.
    return "acme_method ran"


def AcmeNet(num_labels=10, **kwargs):
    # A real model factory returns an nn.Module; a dummy dict here keeps it torch-free.
    return {"kind": "AcmeNet", "num_labels": num_labels}


supreme.register_unlearning_method(
    "acme_method", acme_method
)  # pass the object, not a path
supreme.register_model("AcmeNet", AcmeNet)

from supreme import registry as R

print(
    "method ->", R.resolve_method_location("acme_method")
)  # ('__main__', 'acme_method')
print("model  ->", R.resolve_callable_location("model", "AcmeNet"))


def _load(loc):
    mod, attr = loc
    return getattr(importlib.import_module(mod), attr)


print("call method:", _load(R.resolve_method_location("acme_method"))())
print(
    "call model :", _load(R.resolve_callable_location("model", "AcmeNet"))(num_labels=7)
)

# Registration also syncs the framework's name lists, so the CLI/validation accept them:
print("in all_methods :", "acme_method" in supreme.project_config.all_methods)
print("in model_names :", "AcmeNet" in supreme.project_config.model_names)

## 3. Write your real components

These follow the documented interfaces (see [`docs/extending.md`](../docs/extending.md)).
Define each one **directly in a cell** below - an unlearning method, an evaluation
metric, a model factory and a dataset class. In section 4 you register the **objects
themselves** - no files, no module paths: `register_*` accepts a live callable/class
and resolves it in this process. (In your own project they would live in normal modules
of your package - that is the entry-points route in section 5.) They are torch-based,
so running them needs SUPREME's full stack installed.

### 3a. A custom unlearning method

Signature: the framework always passes `fabric`, `model`, `model_name`,
`num_gpus`, `wandb_logging_flag`, `type_of_unlearning_strategy`, plus the standard
retain/forget train/test dataloaders. Pair every model with its optimiser through
`fabric.setup(...)`, and use `fabric.backward(loss)` for gradient sync.

In [ ]:
import torch
import torch.nn as nn


# An unlearning method receives the Fabric handle, the model to unlearn (an
# independent copy of the original) and the retain/forget dataloaders, plus num_gpus,
# wandb_logging_flag and more (accept **kwargs). It MUST RETURN the unlearned model -
# the framework captures it (compare supreme/methods/unlearning_methods/finetune.py).
def gentle_finetune(
    fabric,
    model,
    retain_train_dataloader,
    num_gpus=1,
    wandb_logging_flag=False,
    lr=0.01,
    epochs=1,
    **kwargs,
):
    """Toy method: fine-tune on the retain set so the model forgets the rest."""
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    # The framework already ran fabric.setup(model); unwrap to the raw module then
    # re-setup it WITH the optimizer (mirrors finetune/ssd/scrub).
    raw_model = model.module if hasattr(model, "module") else model
    model, optimizer = fabric.setup(raw_model, optimizer)
    model.train()
    for _ in range(epochs):
        for images, _, labels in retain_train_dataloader:  # datasets yield (x, _, y)
            optimizer.zero_grad()
            loss = nn.functional.cross_entropy(model(images), labels)
            fabric.backward(loss)  # NOT loss.backward() - Fabric syncs grads
            optimizer.step()
    return model  # <-- the framework uses the returned (unlearned) model

### 3b. A custom evaluation metric

The metric returns a **dict** of named results. Decorate it with
`@track_evaluation_metric` so that dict is wrapped in the standard result envelope
(`{"metric_value_dict": ..., "core_time_dict": ..., ...}`). Passing
`track_evaluation_resources=True` (default `False`) additionally profiles the
metric's **own** time/memory/utilisation - for comparing evaluation metrics
against each other by cost, not for the unlearning-method comparison. Aggregate
across processes with `fabric.all_gather(...)`. Metric names not recognised by the
built-in dispatcher are resolved through the registry and run automatically -
**no edits to `metrics_main.py`**. Pass `requires_retrain=True` at registration
time if the metric needs the retrained reference model.

In [ ]:
import torch
from supreme.utils.unlearning.evaluation_utils import track_evaluation_metric


# A metric returns a DICT of named results. @track_evaluation_metric wraps that in
# the standard envelope; with track_evaluation_resources=True (default False) it also
# profiles the metric's OWN time/memory/compute-utilisation (for comparing the metrics
# by cost). It receives fabric, num_gpus, the model and the retain/forget/test
# dataloaders - use the external-dispatch names (e.g. unlearned_model,
# forget_test_dataloader); compare supreme/eval_metrics/membership_inference_attack.py.
@track_evaluation_metric
def forget_accuracy(
    fabric, num_gpus, unlearned_model, forget_test_dataloader, **kwargs
):
    """Accuracy on the forget set (lower is better after unlearning)."""
    unlearned_model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, _, labels in forget_test_dataloader:
            preds = unlearned_model(images).argmax(dim=1)
            correct += (preds == labels).sum()
            total += labels.numel()
    acc = correct.float() / max(int(total), 1)
    acc = fabric.all_gather(acc).mean().item()
    return {"forget_accuracy": acc}

### 3c. A custom model and dataset

A model is a factory function returning an `nn.Module`. A dataset is a class taking
the framework's constructor kwargs (`root`, `download`, `train`, `unlearning`,
`img_size`, `model_name`); subclassing an existing builder in
`supreme/datasets/datasets.py` is the easiest route. Register the dataset with its
data `root`, the `class_dict` (class-name -> integer label) used by class/subclass
unlearning, and optionally the per-architecture training schedule.

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder


# Model factory: takes num_labels (+ kwargs) and returns an nn.Module - same contract
# as supreme/models/ResNet18.py (def ResNet18(num_labels)) and ViT.py.
def TinyCNN(num_labels, **kwargs):
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(16, num_labels),
    )


# Dataset: subclass a torchvision dataset, take (root, train, unlearning, download,
# img_size, model_name), build the transforms, and have __getitem__ return a 3-TUPLE
# (x, <placeholder>, y) - matches the framework's loader unpacking (`for x, _, y in ...`).
# Compare PinsFaceRecognition / Cifar10 in supreme/datasets/datasets.py.
class MyFaces(ImageFolder):
    def __init__(self, root, train, unlearning, download, img_size=32, model_name=None):
        transform = transforms.Compose(
            [
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
            ]
        )
        super().__init__(root, transform)

    def __getitem__(self, index):
        x, y = super().__getitem__(index)
        return x, torch.Tensor([]), y

## 4. Register everything (runtime API)

Call these once, before launching the pipeline (e.g. at the top of your run script).

In [ ]:
import supreme

# Register the components defined above by passing the OBJECTS directly - no module
# paths, no files. This works because they live in this kernel's __main__ (see the
# section-6 note on in-process vs the grid driver). A dataset accepts a live class too.
supreme.register_unlearning_method("gentle_finetune", gentle_finetune)
supreme.register_metric("forget_accuracy", forget_accuracy, requires_retrain=False)
supreme.register_model("TinyCNN", TinyCNN)
supreme.register_dataset(
    "MyFaces",
    MyFaces,  # a live class, accepted directly
    root="/data/myfaces",
    class_dict={"alice": 0, "bob": 1, "carol": 2},
    rn_epochs=100,
    rn_milestones=[30, 60, 80],  # ResNet schedule
    vit_epochs=8,
    vit_milestones=[7],  # ViT schedule
)

pc = supreme.project_config
print("method  registered:", "gentle_finetune" in pc.all_methods)
print("metric  registered:", "forget_accuracy" in pc.evaluation_metrics)
print("model   registered:", "TinyCNN" in pc.model_names)
print("dataset registered:", "MyFaces" in pc.dataset_names)

## 5. Alternative: ship a plugin via packaging entry points

Instead of calling `register_*` at runtime, an installed package can declare its
components in its packaging metadata; SUPREME discovers them automatically on first
use. Use the direct `module:attribute` groups for the callable categories:

```toml
# in your plugin package's pyproject.toml
[project.entry-points."supreme.unlearning_methods"]
gentle_finetune = "my_extras_method:gentle_finetune"

[project.entry-points."supreme.metrics"]
forget_accuracy = "my_extras_metric:forget_accuracy"

[project.entry-points."supreme.models"]
TinyCNN = "my_extras_model:TinyCNN"
```

Datasets carry extra metadata (root, class dict, schedule), and you may want to
register several components at once, so point the `supreme.plugins` group at a
zero-argument **setup callable** that registers them via the runtime API:

```toml
[project.entry-points."supreme.plugins"]
my_extras = "my_extras_register:setup"
```

The `supreme.plugins` setup callable lives in a normal module of your package, e.g.
`my_extras/register.py`:

```python
import supreme
from my_extras.method import gentle_finetune
from my_extras.dataset import MyFaces

def setup():
    """Discovered via the 'supreme.plugins' entry-point group on first use."""
    supreme.register_unlearning_method("gentle_finetune", gentle_finetune)
    supreme.register_dataset("MyFaces", MyFaces, root="/data/myfaces",
                             class_dict={"alice": 0, "bob": 1, "carol": 2})
```

Unlike the runtime registration in sections 3-4 (which lives in *this* notebook's
process), an installed plugin is importable in **any** process - so it also works with
the `run_local.sh` / SLURM drivers that spawn subprocesses, not just in-process.

## 6. Run the pipeline with your components

Pass your registered names to `run_training` / `run_unlearning` (or the
`supreme-train` / `supreme-unlearn` console scripts) just like the built-in names.
The cell below runs the pipeline in-process (train -> unlearn -> evaluate) with your
custom model, method, and metric on the built-in Cifar10 dataset (auto-downloads).
Your own dataset (e.g. `MyFaces`) works the same way once its data is in place.

**In-process vs the grid driver.** Runtime `register_*` (above) lives in *this* process, so
it applies to in-process `run_training`/`run_unlearning`. The shell driver `run_local.sh`
spawns fresh subprocesses that don't see runtime registration - for that route, ship your
components as an installed plugin via packaging entry points (section 5).

> Small run: trains on MPS/CPU, logs to W&B offline (so it doesn't touch your cloud
> account), epochs shrunk to 1. It checks the wiring end-to-end, not paper accuracy.

In [ ]:
# Full pipeline (train -> unlearn -> evaluate) with YOUR registered components,
# IN-PROCESS so the runtime register_* calls above apply. Uses built-in Cifar10
# (auto-downloads) so it is self-contained. Tiny by design (epochs=1).
import os, glob, multiprocessing as mp

mp.set_start_method("fork", force=True)  # macOS DataLoader workers (notebook-safe)
os.environ.update(
    {
        "WANDB_MODE": "offline",
        "TRAINING_SEED": "260",
        "UNLEARNING_SEED": "260",
        "WANDB_PROJECT_PREFIX": "DEMO",
    }
)

import supreme

pc = supreme.project_config
pc.Cifar10_RN_EPOCHS = 1  # tiny smoke (real default is 20); proves wiring
pc.Cifar10_RN_MILESTONES = []

# 1) train the custom model
supreme.run_training(
    [
        "-unlearning_seed",
        "260",
        "-precision",
        "32-true",
        "-net",
        "TinyCNN",
        "-dataset",
        "Cifar10",
        "-classes",
        "10",
        "-unlearning_context",
        "random",
        "-include_gpus_in_path",
        "true",
        "-training_seed",
        "260",
    ]
)

# 2) locate the checkpoint just written (the driver does this via update_checkpoint_paths.py)
ckpt = sorted(
    glob.glob(
        os.path.join(
            pc.CHECKPOINT_PATH,
            "precision_32-true/1gpus/no_dist/train_seed_260/unlearning_seed_260/"
            "model_checkpoints/TinyCNN/Cifar10/*/*-best.pth",
        )
    )
)[-1]

# 3) per-cell unlearning output dir (run_local.sh exports this as LOG_DIR)
log_dir = os.path.join(
    pc.PROJECT_ROOT,
    "logs/unlearning/precision_32-true/1gpus/"
    "train_seed_260/unlearn_seed_260/random_/Cifar10/TinyCNN/classes_10/forget_perc_0.01",
)
os.makedirs(log_dir, exist_ok=True)
os.environ["LOG_DIR"] = log_dir

# 4) unlearn (gentle_finetune), then evaluate (accuracy + your forget_accuracy).
#    unlearn_main runs in two env-toggled phases via PERFORM_EVALUATION.
args = [
    "-net",
    "TinyCNN",
    "-dataset",
    "Cifar10",
    "-type_of_unlearning_strategy",
    "random_",
    "-weight_path",
    ckpt,
    "-seed",
    "260",
    "-precision",
    "32-true",
    "-eval_metrics",
    "accuracy,forget_accuracy",
    "-classes",
    "10",
    "-forget_perc",
    "0.01",
]
supreme.run_unlearning(["-method", "gentle_finetune", *args])  # unlearning stage
os.environ["PERFORM_EVALUATION"] = "true"
supreme.run_unlearning(["-method", "gentle_finetune", *args])  # evaluation stage
print(
    "Done: trained TinyCNN, unlearned with gentle_finetune, evaluated accuracy + forget_accuracy."
)